# 4장 — 사람의 취향을 숫자로 🏅 (핵심)

웹 챕터의 약속 **비교(A/B)를 먹이면 점수(r)가 태어난다**를 코드로 지킵니다.

이번엔 흉내가 아니라 **진짜 경사하강**(2부 9장)입니다:
1. 답변 4개에 점수 r 을 두고(전부 0에서 시작)
2. 사람의 선호 쌍으로 손실 −log σ(r_선택 − r_탈락) 을 재고
3. 기울기를 따라 점수를 갱신 → **선호 순위가 점수에 새겨지는 것**을 확인합니다

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch4')

## 1. 재료: 답변 4개와 사람의 선호

"회의록을 세 줄로 요약해 줘"에 대한 답변 후보 4개입니다.
사람 라벨러들이 쌍끼리 비교한 결과(선호 쌍)가 데이터의 전부입니다. **점수는 어디에도 없어요**.

In [ ]:
import math

ANSWERS = {
    'A': '① 일정 확정 ② 담당 배정 ③ 다음 회의 금요일  (간결·정확)',
    'B': '네, 요약해 드리겠습니다. 회의에서는 먼저…  (정확하지만 장황)',
    'C': '요약 불가. 회의록이 너무 김.  (무성의)',
    'D': '이 회의록을 세 줄로 요약해 줘 라는 요청을 받았을 때는…  (이어쓰기 증상)',
}

# 사람의 선호 쌍: (선택된 쪽, 탈락한 쪽) — 라벨러 체험(웹 실험 ①)의 결과물이라고 상상
PREFS = [
    ('A', 'B'), ('A', 'C'), ('A', 'D'),
    ('B', 'C'), ('B', 'D'),
    ('C', 'D'),
    ('A', 'B'), ('B', 'C'),  # 몇 쌍은 여러 라벨러가 중복 평가
]

# 보상 점수 — 전부 0에서 시작 (모델은 아직 취향을 모른다)
r = {k: 0.0 for k in ANSWERS}
print('초기 점수:', r)

## 2. Bradley-Terry 손실과 기울기

- 선호 확률: **p = σ(r_선택 − r_탈락)** (시그모이드, 2부 7장)
- 손실: **−log p** (또 크로스 엔트로피 패턴, 5부 4장)
- 기울기(미분해 보면 아주 단순해요): 손실을 줄이려면 **r_선택 은 (1−p) 만큼 올리고, r_탈락 은 (1−p) 만큼 내리면** 됩니다.
  - p 가 낮을수록(사람과 반대로 생각할수록) (1−p) 가 커서 크게 교정: "크게 놀란 만큼 크게 배움" 그대로!

In [ ]:
def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def avg_loss():
    return sum(-math.log2(sigmoid(r[w] - r[l])) for w, l in PREFS) / len(PREFS)

LR = 0.5  # 학습률 (2부 9장 — 한 걸음의 보폭)

def train_step():
    """진짜 경사하강 한 걸음 — 선호 쌍마다 r_선택 ↑, r_탈락 ↓ (크기는 1−p 에 비례)."""
    for w, l in PREFS:
        p = sigmoid(r[w] - r[l])
        r[w] += LR * (1 - p)
        r[l] -= LR * (1 - p)

curve = [avg_loss()]
for _ in range(30):
    train_step()
    curve.append(avg_loss())

print(f'훈련 전 평균 손실: {curve[0]:.2f}비트  (σ(0)=0.5 → −log₂(0.5) = 1비트)')
print(f'훈련 후 평균 손실: {curve[-1]:.2f}비트')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3))
plt.plot(range(len(curve)), curve, marker='o', markersize=3, color='#6366f1')
plt.xlabel('훈련 스텝')
plt.ylabel('평균 손실 (비트)')
plt.title('보상 모델 학습 곡선: 취향이 점수로 압축되는 중')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 태어난 점수 확인: 취향의 디지털화

점수는 어디에도 입력한 적이 없습니다. 비교만 먹였는데 어떻게 됐을까요?

In [ ]:
ranking = sorted(r, key=r.get, reverse=True)
print('학습된 보상 점수 (높은 순):')
for k in ranking:
    print(f'   {k}: r = {r[k]:+.2f}   {ANSWERS[k]}')

pAB = sigmoid(r['A'] - r['B'])
print(f"\n이제 새 비교도 예측 가능: P(A가 B보다 선호) = σ(r_A − r_B) = {pAB:.0%}")

# 검증 — 선호 순위가 점수에 새겨졌는가
assert curve[-1] < curve[0], '학습으로 손실이 내려가야 한다!'
assert ranking == ['A', 'B', 'C', 'D'], '점수 순위가 사람의 선호(A>B>C>D)와 일치해야 한다!'
assert pAB > 0.5, '선호된 쪽의 승률 예측이 50%를 넘어야 한다!'
print('\n✔ 확인: 비교만 먹였는데 점수와 순위가 태어났다 (취향의 디지털화)')

## 4. 관찰 정리 + 🤖 AI 연결

- **σ(점수 차) = 선호 확률**(2부 7장 시그모이드 · 3부 4장 조건부 확률), **손실 = −log p**(5부 4장 패턴),
  **갱신 = 경사하강**(2부 9장). 전부 아는 얼굴의 조립.
- (1−p) 만큼 교정: 사람과 반대로 생각한 쌍일수록 크게 배우는 구조가 수식에 내장돼 있었죠.
- 진짜 보상 모델과 다른 점: 진짜는 답변마다 점수 변수를 두는 게 아니라, **답변 텍스트를 읽고**
  점수를 내는 신경망입니다(LLM의 마지막 층 교체). 그래서 처음 보는 답변에도 점수를 매길 수 있어요.

**다음 챕터**: 5장, 이 점수를 보상 삼아 모델을 움직이는 법, 강화학습의 최소 직관.

> 💡 궁금한 코드는 그 셀을 실행한 다음 셀에서 `from tutor import explain` 후 `explain('ch4')`.